In [1]:
import os
os.environ["MLFLOW_TRACKING_URI"] = "file:../mlruns"

import mlflow
print(mlflow.get_tracking_uri())  

file:../mlruns


In [2]:
import xgboost as xgb
print(xgb.__version__)

3.0.4


In [3]:
import sys, xgboost as xgb
print(sys.executable)        # should point to .../.venv/bin/python
print(xgb.__version__)       # should print 3.0.4
print(xgb.__file__)          # should live under .../.venv/...

/home/khaipd18/Housing-MLOps/.venv/bin/python
3.0.4
/home/khaipd18/Housing-MLOps/.venv/lib/python3.11/site-packages/xgboost/__init__.py


In [4]:
# ==============================================
# 1. Imports
# ==============================================
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor
import optuna
import mlflow
import mlflow.xgboost

In [5]:
# ==============================================
# 2. Load processed datasets
# ==============================================
train_df = pd.read_csv(r"../data/processed/feature_engineered_train.csv")
eval_df  = pd.read_csv(r"../data/processed/feature_engineered_eval.csv")


# Define target + features
target = "price"
X_train, y_train = train_df.drop(columns=[target]), train_df[target]
X_eval, y_eval   = eval_df.drop(columns=[target]), eval_df[target]

print("Train shape:", X_train.shape)
print("Eval shape:", X_eval.shape)

Train shape: (576815, 39)
Eval shape: (148448, 39)


In [6]:
# ==============================================
# 3. Define Optuna objective function with MLflow
# ==============================================
def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 200, 1000),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "gamma": trial.suggest_float("gamma", 0.0, 5.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        "random_state": 42,
        "n_jobs": -1,
        "tree_method": "hist",
    }

    with mlflow.start_run(nested=True):
        model = XGBRegressor(**params)
        model.fit(X_train, y_train)

        y_pred = model.predict(X_eval)
        rmse = float(np.sqrt(mean_squared_error(y_eval, y_pred)))
        mae = float(mean_absolute_error(y_eval, y_pred))
        r2 = float(r2_score(y_eval, y_pred))

        # Log hyperparameters + metrics
        mlflow.log_params(params)
        mlflow.log_metrics({"rmse": rmse, "mae": mae, "r2": r2})

    return rmse

In [7]:
# ==============================================
# 4. Run Optuna study with MLflow
# ==============================================
# Force MLflow to always use the root project mlruns folder
#mlflow.set_tracking_uri("file:///mnt/d/Github/Development-and-Deployment-of-a-Housing-Price-Prediction-System-using-MLOps/mlruns")
mlflow.set_tracking_uri(os.environ["MLFLOW_TRACKING_URI"])
mlflow.set_experiment("xgboost_optuna_housing")

study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=15)

print("Best params:", study.best_trial.params)

2026/06/04 16:33:20 INFO mlflow.tracking.fluent: Experiment with name 'xgboost_optuna_housing' does not exist. Creating a new experiment.
[I 2026-06-04 16:33:20,193] A new study created in memory with name: no-name-edcec19c-facb-4f86-9874-d77bd4d9fd09


[I 2026-06-04 16:38:44,697] Trial 0 finished with value: 72617.24920326956 and parameters: {'n_estimators': 830, 'max_depth': 7, 'learning_rate': 0.11704556864991919, 'subsample': 0.8895035329162264, 'colsample_bytree': 0.8473786043294105, 'min_child_weight': 10, 'gamma': 4.00552367757594, 'reg_alpha': 0.8788440201370016, 'reg_lambda': 1.1482235098905272e-06}. Best is trial 0 with value: 72617.24920326956.
[I 2026-06-04 16:42:23,840] Trial 1 finished with value: 77739.11762695372 and parameters: {'n_estimators': 889, 'max_depth': 5, 'learning_rate': 0.012167274987829648, 'subsample': 0.8305827286686813, 'colsample_bytree': 0.6822995138565715, 'min_child_weight': 2, 'gamma': 2.330033681780457, 'reg_alpha': 6.186200454441742, 'reg_lambda': 0.03651694567189921}. Best is trial 0 with value: 72617.24920326956.
[I 2026-06-04 16:48:24,544] Trial 2 finished with value: 70315.08521000847 and parameters: {'n_estimators': 943, 'max_depth': 9, 'learning_rate': 0.014567457188998935, 'subsample': 0.

Best params: {'n_estimators': 711, 'max_depth': 9, 'learning_rate': 0.010387050590858473, 'subsample': 0.9856433543709908, 'colsample_bytree': 0.5071180420098788, 'min_child_weight': 8, 'gamma': 1.2374100576778269, 'reg_alpha': 0.022910711567182624, 'reg_lambda': 7.983691837111384}


In [8]:
# ==============================================
# 5. Train final model with best params and log to MLflow
# ==============================================
best_params = study.best_trial.params
best_model = XGBRegressor(**best_params)
best_model.fit(X_train, y_train)

y_pred = best_model.predict(X_eval)

mae = mean_absolute_error(y_eval, y_pred)
rmse = np.sqrt(mean_squared_error(y_eval, y_pred))
r2 = r2_score(y_eval, y_pred)

print("Final tuned model performance:")
print("MAE:", mae)
print("RMSE:", rmse)
print("R²:", r2)

# Log final model
with mlflow.start_run(run_name="best_xgboost_model"):
    mlflow.log_params(best_params)
    mlflow.log_metrics({"rmse": rmse, "mae": mae, "r2": r2})
    print(mlflow.get_tracking_uri())
    print(mlflow.get_artifact_uri())
    mlflow.xgboost.log_model(best_model, name="model")

Final tuned model performance:
MAE: 31922.60602940347
RMSE: 68601.18408131381
R²: 0.9636316787283099
file:../mlruns
file:///home/khaipd18/Housing-MLOps/notebooks/../mlruns/884712366487792673/85dde28873fb40ebaf3d90fd32920d14/artifacts


/home/khaipd18/Housing-MLOps/.venv/lib/python3.11/site-packages/xgboost/sklearn.py:1028: UserWarning: [17:36:25] WARNING: /workspace/src/c_api/c_api.cc:1427: Saving model in the UBJSON format as default.  You can use file extension: `json`, `ubj` or `deprecated` to choose between formats.
  self.get_booster().save_model(fname)
2026/06/04 17:36:29 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
2026/06/04 17:36:29 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
